# Ilera-ATLAS · 04_eval: score a model on Ilera-Bench and AfriMed-QA

One notebook for every evaluation in the plan:

| When | Runbook | Config | Output (HF `ilera-data/results/`) |
|---|---|---|---|
| **Mon 5 Oct** | Part 7 baseline | `MODEL = "NCAIR1/N-ATLaS"`, `TAG = "base"`, `EVAL_SET = "bench"` | `scores_base.csv` |
| **Wed 7 Oct** | Part 8.4 pick the best run | `MODEL = "<you>/ilera-atlas-lora-runA"` (B, C...), `TAG = "runA"`, `EVAL_SET = "val"` | `val_runA.csv` |
| **Thu 8 Oct** | Part 9 final | `MODEL = "<you>/ilera-atlas-lora-runB"`, `TAG = "ilera"`, `EVAL_SET = "bench"`, `COMPARE_WITH = "base"`, `MAKE_BLIND_SHEET = True` | `scores_ilera.csv`, `compare_base_vs_ilera.csv`, blind-review sheet |

**Fairness rules (do not change between models):** same `SYSTEM` prompt (loaded from
`config/system_prompt.txt`), greedy decoding, `MAX_NEW = 450`, `repetition_penalty = 1.1`, 4-bit loading.
Base N-ATLaS also gets the system prompt, so any gain comes from fine-tuning, not from a hidden instruction.

**Never pick a training run with `EVAL_SET = "bench"`.** Use `"val"` to choose; run the benchmark once
on the chosen model.

**Kaggle:** GPU T4 x2 · Internet On · secret `HF_TOKEN`. Smoke test with `LIMIT = 16` first.
Generation is resumable: re-running continues where it stopped.

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"           # <- change this
DATA_REPO = f"{HF_USER}/ilera-data"
MODEL = "NCAIR1/N-ATLaS"               # base model, or your LoRA repo e.g. f"{HF_USER}/ilera-atlas-lora-runB"
TAG = "base"                           # short name for this model in result files
EVAL_SET = "bench"                     # "bench" (Ilera-Bench + AfriMed-QA) or "val" (validation set)
COMPARE_WITH = None                    # e.g. "base" when scoring the fine-tuned model
MAKE_BLIND_SHEET = False               # True on Thu 8 Oct: sheet for the blind clinical review
N_BLIND = 150
LIMIT = 16                             # smoke test: first N items. Set None for the full run.
BATCH = 8
MAX_NEW = 450
SAVE_EVERY = 10                        # batches between progress uploads
N_BOOT = 2000                          # bootstrap resamples for confidence intervals
USE_LID = True                         # GlotLID language check (1.6 GB download once)
UPLOAD = True
LOCAL_DIR = None                       # offline testing: folder mirroring the HF dataset layout

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import subprocess, sys
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)   # deprecated Kaggle default -> removes the FutureWarning
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"         # its replacement: faster model downloads
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth", "fasttext-wheel"], capture_output=True, text=True)
if r.returncode != 0:                               # pip prints harmless dependency warnings; show them only on failure
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install failed")
print("packages installed")
# If the next cells fail with an import error: Run > Restart kernel, then Run All again.

In [ ]:
import os
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
if UPLOAD or not LOCAL_DIR:
    assert HF_TOKEN, "No HF_TOKEN found: add it as a Kaggle/Colab secret or env var"
    os.environ["HF_TOKEN"] = HF_TOKEN

## 1. Helpers and data

In [ ]:
import csv, hashlib, io, json, random, re, time, collections, urllib.request
import numpy as np

def fetch(path, required=True):
    if LOCAL_DIR:
        p = os.path.join(LOCAL_DIR, path)
        if os.path.exists(p):
            return p
        if required:
            raise FileNotFoundError(p)
        return None
    try:
        from huggingface_hub import hf_hub_download
        return hf_hub_download(DATA_REPO, path, repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        if required:
            raise FileNotFoundError(f"{path} not found in {DATA_REPO}: {e}")
        return None

def upload(files, msg):
    if not UPLOAD:
        print("(dry run) would upload:", [r for _, r in files])
        return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for local, remote in files:
        api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=DATA_REPO,
                        repo_type="dataset", commit_message=msg)

def read_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

SYSTEM = open(fetch("config/system_prompt.txt"), encoding="utf-8").read().strip()
os.makedirs("results", exist_ok=True)

if EVAL_SET == "bench":
    bpath = fetch("bench/ilera_bench.jsonl")
    recorded = dict(l.split()[::-1] for l in open(fetch("bench/BENCH_SHA256.txt")) if l.strip())
    assert hashlib.sha256(open(bpath, "rb").read()).hexdigest() == recorded["ilera_bench.jsonl"], \
        "ilera_bench.jsonl does not match BENCH_SHA256.txt: the frozen benchmark was changed"
    items = read_jsonl(bpath)
    print(f"Ilera-Bench (hash OK): {len(items)} items",
          dict(collections.Counter((x['subset'], x['lang']) for x in items)))
elif EVAL_SET == "val":
    items = []
    for m in read_jsonl(fetch("val.jsonl")):
        gold = m["messages"][-1]["content"]
        g = re.search(r"ACTION:\s*(TREAT|REFER|URGENT)", gold)
        if not g:
            continue
        items.append({"id": m["id"], "subset": "val", "lang": m["lang"],
                      "question": m["messages"][-2]["content"], "gold_action": g.group(1),
                      "key_facts": [], "gold_answer": gold, "source": ""})
    print(f"validation set: {len(items)} items", dict(collections.Counter(x['lang'] for x in items)))
else:
    raise ValueError("EVAL_SET must be 'bench' or 'val'")
if LIMIT:
    items = items[:LIMIT]
    print(f"*** SMOKE TEST: only {LIMIT} items. Set LIMIT = None for the full run. ***")

## 2. AfriMed-QA multiple-choice items (bench only)
Reloaded from the public AfriMed-QA CSV using the frozen `bench/afrimedqa_ids.txt`. Options with
`N/A` are dropped; options are relabelled A–E. Both models get the same neutral prompt **without**
the Ilera system prompt (that prompt asks for the ACTION format, not a letter).

In [ ]:
AFRIMED_URL = ("https://raw.githubusercontent.com/intron-innovation/AfriMed-QA/master/data/"
               "afri_med_qa_15k_v2.5_phase_2_15275.csv")
mcq = []
if EVAL_SET == "bench":
    ids = [l.strip() for l in open(fetch("bench/afrimedqa_ids.txt")) if l.strip()]
    req = urllib.request.Request(AFRIMED_URL, headers={"User-Agent": "Mozilla/5.0"})
    rows = {r["sample_id"]: r for r in csv.DictReader(io.StringIO(
        urllib.request.urlopen(req, timeout=300).read().decode("utf-8")))}
    for sid in ids:
        r = rows[sid]
        opts = [(k, v) for k, v in sorted(json.loads(r["answer_options"]).items())
                if v and v.strip().upper() != "N/A"]
        letters = "ABCDE"[:len(opts)]
        gold = letters[[k for k, _ in opts].index(r["correct_answer"])]
        prompt = (r["question_clean"] or r["question"]).strip() + "\n\n" + \
                 "\n".join(f"{l}. {v}" for l, (_, v) in zip(letters, opts)) + \
                 "\n\nAnswer with the letter of the single best option only."
        mcq.append({"id": sid, "prompt": prompt, "gold": gold, "letters": letters,
                    "specialty": r["specialty"], "country": r["country"]})
    if LIMIT:
        mcq = mcq[:LIMIT]
    print(len(mcq), "AfriMed-QA MCQ items", dict(collections.Counter(m["specialty"] for m in mcq)))

## 3. Load the model and `chat()`

In [ ]:
import torch
from unsloth import FastLanguageModel

# Works for the base model and for a LoRA repo (Unsloth loads N-ATLaS + the adapter).
model, tok = FastLanguageModel.from_pretrained(MODEL, max_seq_length=2048, load_in_4bit=True, token=HF_TOKEN)
FastLanguageModel.for_inference(model)
model.generation_config.max_length = None           # avoids the "max_new_tokens and max_length" warning
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def chat(user_msgs, system=None, max_new=MAX_NEW):
    texts = []
    for u in user_msgs:
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": u}]
        texts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]

## 4. Generate answers (resumable)

In [ ]:
def generate(name, jobs, system, max_new):
    """jobs: list of (id, prompt). Results cached in results/{name}.jsonl, also on HF."""
    path = f"results/{name}.jsonl"
    if not os.path.exists(path) and not LOCAL_DIR:
        p = fetch(path, required=False)
        if p:
            import shutil; shutil.copy(p, path); print("resuming from Hugging Face copy of", path)
    done = {r["id"]: r["output"] for r in read_jsonl(path)} if os.path.exists(path) else {}
    todo = sorted([j for j in jobs if j[0] not in done], key=lambda j: len(j[1]))
    print(f"{name}: {len(jobs)} items | {len(done)} cached | {len(todo)} to generate")
    t0, nb = time.time(), (len(todo) + BATCH - 1) // BATCH
    with open(path, "a", encoding="utf-8") as f:
        for b in range(nb):
            batch = todo[b * BATCH:(b + 1) * BATCH]
            for (i, _), o in zip(batch, chat([p for _, p in batch], system=system, max_new=max_new)):
                done[i] = o
                f.write(json.dumps({"id": i, "output": o}, ensure_ascii=False) + "\n")
            f.flush()
            if (b + 1) % 5 == 0 or b + 1 == nb:
                el = time.time() - t0
                print(f"  batch {b + 1}/{nb}  {el / 60:.1f} min  ETA {el / (b + 1) * (nb - b - 1) / 60:.0f} min", flush=True)
            if (b + 1) % SAVE_EVERY == 0:
                upload([(path, path)], f"04_eval {name}: progress")
    upload([(path, path)], f"04_eval {name}: done")
    return done

prefix = "outputs" if EVAL_SET == "bench" else "val_outputs"
outputs = generate(f"{prefix}_{TAG}", [(x["id"], x["question"]) for x in items], SYSTEM, MAX_NEW)
mcq_out = generate(f"afrimed_{TAG}", [(m["id"], m["prompt"]) for m in mcq], None, 20) if mcq else {}

## 5. Scoring
| Metric | Meaning | Better |
|---|---|---|
| `action_ok` | ACTION matches the gold label | higher |
| `under_referral` | model says a lower level than gold (e.g. TREAT when gold is REFER/URGENT), **or gives no ACTION at all**: the dangerous error | lower |
| `over_referral` | model says a higher level than gold (safe but wasteful) | lower |
| `format_ok` | answer has a valid ACTION line and a SOURCE line | higher |
| `key_fact_recall` | share of the item's key facts found in the answer | higher |
| `source_ok` | SOURCE line cites the right Standing Orders section | higher |
| `same_language` | answer is in the question's language (GlotLID) | higher |
| `mcq_acc` | AfriMed-QA letter correct | higher |

95% confidence intervals: bootstrap over items. Comparisons between two models use a **paired**
bootstrap (same items resampled for both), which is the right test for "is model B better than A".

In [ ]:
lid = None
if USE_LID:
    try:
        import fasttext
        from huggingface_hub import hf_hub_download
        lid = fasttext.load_model(hf_hub_download("cis-lmu/glotlid", "model.bin"))
    except Exception as e:
        print("GlotLID unavailable, same_language not scored:", e)
LID = {"en": "__label__eng_Latn", "ha": "__label__hau_Latn", "yo": "__label__yor_Latn", "ig": "__label__ibo_Latn"}
RANK = {"TREAT": 0, "REFER": 1, "URGENT": 2}

def action_of(text):
    m = re.search(r"ACTION\s*:\s*\**\s*(TREAT|REFER|URGENT)", text.upper())
    return m.group(1) if m else None

def section_of(text):
    m = re.search(r"Section\s+(\d{1,2}(?:\.\d{1,2}){1,2})", text or "")
    return m.group(1) if m else None

def score_item(x, out):
    pred, gold = action_of(out), x["gold_action"]
    src_line = re.search(r"SOURCE:\s*(.*)", out)
    s = {"id": x["id"], "subset": x["subset"], "lang": x["lang"], "pred_action": pred,
         "action_ok": float(pred == gold),
         "under_referral": float(pred is None or RANK[pred] < RANK[gold]),
         "over_referral": float(pred is not None and RANK[pred] > RANK[gold]),
         "format_ok": float(pred is not None and src_line is not None),
         "key_fact_recall": (sum(f.lower() in out.lower() for f in x["key_facts"]) / len(x["key_facts"])
                             if x["key_facts"] else np.nan),
         "source_ok": (float(src_line is not None and section_of(src_line.group(1)) == section_of(x["source"]))
                       if section_of(x.get("source")) else np.nan),
         "same_language": np.nan}
    if lid is not None and len(out.split()) >= 4:
        body = re.sub(r"(ACTION|WHAT TO DO|DANGER SIGNS|SOURCE):.*?(\n|$)", " ", out)   # labels are English by design
        if len(body.split()) >= 4:
            s["same_language"] = float(lid.f.predict(body.replace("\n", " ")[:600], 1, 0.0, "strict")[0][1] == LID[x["lang"]])
    return s

def letter_of(out, letters):
    m = re.search(rf"\b([{letters}])\b", out.strip().upper()[:40])
    return m.group(1) if m else None

# self-check
_x = {"id": "t", "subset": "main", "lang": "en", "gold_action": "REFER", "key_facts": ["amoxicillin", "5 days"],
      "source": "CHEW Standing Orders 2024, Section 2.3 Fever, p. 50"}
_s = score_item(_x, "ACTION: TREAT\nWHAT TO DO:\n1. Give amoxicillin.\nSOURCE: CHEW Standing Orders 2024, Section 2.3 Fever")
assert _s["action_ok"] == 0 and _s["under_referral"] == 1 and _s["key_fact_recall"] == 0.5 and _s["source_ok"] == 1
assert letter_of("B. Simple goiter", "ABCDE") == "B" and letter_of("The answer is D", "ABCD") == "D"
print("scoring self-check passed")

In [ ]:
METRICS = ["action_ok", "under_referral", "over_referral", "format_ok", "key_fact_recall", "source_ok", "same_language"]
rng = np.random.default_rng(0)

def ci(vals):
    v = np.array([x for x in vals if not np.isnan(x)], dtype=float)
    if len(v) == 0:
        return (np.nan, np.nan, np.nan, 0)
    boots = rng.choice(v, (N_BOOT, len(v))).mean(1)
    return (v.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5), len(v))

scored = [score_item(x, outputs[x["id"]]) for x in items]
for s in scored:
    s["output"] = outputs[s["id"]]
groups = collections.defaultdict(list)
for s in scored:
    groups[(s["subset"], s["lang"])].append(s)
    if s["subset"] == "main":
        groups[("main", "ALL")].append(s)

table = []
for (subset, lang), ss in sorted(groups.items()):
    for m in METRICS:
        mean, lo, hi, n = ci([s[m] for s in ss])
        if n:
            table.append({"tag": TAG, "subset": subset, "lang": lang, "metric": m, "mean": round(mean, 4),
                          "ci_low": round(lo, 4), "ci_high": round(hi, 4), "n": n})
if mcq_out:
    mcq_scored = [{"id": m["id"], "specialty": m["specialty"], "gold": m["gold"],
                   "pred": letter_of(mcq_out[m["id"]], m["letters"])} for m in mcq]
    mean, lo, hi, n = ci([float(r["pred"] == r["gold"]) for r in mcq_scored])
    table.append({"tag": TAG, "subset": "afrimedqa", "lang": "en", "metric": "mcq_acc", "mean": round(mean, 4),
                  "ci_low": round(lo, 4), "ci_high": round(hi, 4), "n": n})
    print(f"AfriMed-QA: {mean:.3f} [{lo:.3f}-{hi:.3f}]  unparsed answers: {sum(r['pred'] is None for r in mcq_scored)}")

name = f"scores_{TAG}" if EVAL_SET == "bench" else f"val_{TAG}"
with open(f"results/{name}.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(table[0])); w.writeheader(); w.writerows(table)
write_jsonl(f"results/scored_{name}.jsonl", scored)
upload([(f"results/{name}.csv", f"results/{name}.csv"), (f"results/scored_{name}.jsonl", f"results/scored_{name}.jsonl")],
       f"04_eval: {name}")

print(f"\n{TAG} on {EVAL_SET}:")
print(f"{'subset':8} {'lang':5} " + " ".join(f"{m[:14]:>15}" for m in METRICS))
for (subset, lang), ss in sorted(groups.items()):
    row = {t["metric"]: t for t in table if t["subset"] == subset and t["lang"] == lang}
    print(f"{subset:8} {lang:5} " + " ".join(f"{row[m]['mean']:>15.3f}" if m in row else f"{'-':>15}" for m in METRICS))

## 6. Compare with another model (paired bootstrap)
Set `COMPARE_WITH = "base"` when scoring the fine-tuned model. A difference is reported as
**significant** when its 95% CI does not include 0. For `under_referral` and `over_referral` a
**negative** difference is an improvement.

In [ ]:
if COMPARE_WITH and EVAL_SET == "bench":
    other_path = f"results/scored_scores_{COMPARE_WITH}.jsonl"
    if not os.path.exists(other_path):
        p = fetch(other_path)
        import shutil; shutil.copy(p, other_path)
    A = {s["id"]: s for s in read_jsonl(other_path)}
    B = {s["id"]: s for s in scored}
    common = [i for i in B if i in A]
    print(f"paired comparison on {len(common)} items: {COMPARE_WITH} (A) vs {TAG} (B)")
    comp = []
    keys = sorted({(B[i]["subset"], B[i]["lang"]) for i in common}) + [("main", "ALL")]
    for subset, lang in keys:
        ids = [i for i in common if B[i]["subset"] == subset and (lang == "ALL" or B[i]["lang"] == lang)]
        for m in METRICS:
            pairs = np.array([(A[i][m], B[i][m]) for i in ids
                              if A[i][m] is not None and B[i][m] is not None
                              and not (np.isnan(A[i][m]) or np.isnan(B[i][m]))], dtype=float)
            if len(pairs) == 0:
                continue
            d = pairs[:, 1] - pairs[:, 0]
            boots = rng.choice(d, (N_BOOT, len(d))).mean(1)
            lo, hi = np.percentile(boots, [2.5, 97.5])
            comp.append({"subset": subset, "lang": lang, "metric": m, COMPARE_WITH: round(pairs[:, 0].mean(), 4),
                         TAG: round(pairs[:, 1].mean(), 4), "diff": round(d.mean(), 4),
                         "ci_low": round(lo, 4), "ci_high": round(hi, 4), "n": len(d),
                         "significant": bool(lo > 0 or hi < 0)})
    cname = f"results/compare_{COMPARE_WITH}_vs_{TAG}.csv"
    with open(cname, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(comp[0])); w.writeheader(); w.writerows(comp)
    upload([(cname, cname)], f"04_eval: compare {COMPARE_WITH} vs {TAG}")
    print(f"{'subset/lang':14} {'metric':16} {COMPARE_WITH:>8} {TAG:>8} {'diff':>8}  95% CI")
    for c in comp:
        if c["lang"] == "ALL" or c["metric"] in ("action_ok", "under_referral"):
            print(f"{c['subset'] + '/' + c['lang']:14} {c['metric']:16} {c[COMPARE_WITH]:>8.3f} {c[TAG]:>8.3f} "
                  f"{c['diff']:>+8.3f}  [{c['ci_low']:+.3f}, {c['ci_high']:+.3f}]{'  *' if c['significant'] else ''}")

## 7. Blind clinical-review sheet (Thu 8 Oct)
`N_BLIND` random main-benchmark items. For each item the two answers appear as **Answer 1 / Answer 2**
in random order. The reviewer grades each answer **correct / partly correct / wrong / dangerous** and
picks a preferred answer. The key (which answer is which model) is a separate file: do **not** give it
to the reviewer; un-blind only after grading.

In [ ]:
if MAKE_BLIND_SHEET and COMPARE_WITH and EVAL_SET == "bench":
    other_out = {r["id"]: r["output"] for r in read_jsonl(fetch(f"results/outputs_{COMPARE_WITH}.jsonl")
                 if not os.path.exists(f"results/outputs_{COMPARE_WITH}.jsonl") else f"results/outputs_{COMPARE_WITH}.jsonl")}
    pool = [x for x in items if x["subset"] == "main" and x["id"] in other_out]
    pick = random.Random(42).sample(pool, min(N_BLIND, len(pool)))
    sheet, key = [], []
    for k, x in enumerate(pick):
        a, b = (other_out[x["id"]], COMPARE_WITH), (outputs[x["id"]], TAG)
        first, second = (a, b) if random.Random(1000 + k).random() < 0.5 else (b, a)
        sheet.append({"row": k + 1, "item_id": x["id"], "lang": x["lang"], "question": x["question"],
                      "reference_answer": x["gold_answer"], "answer_1": first[0], "answer_2": second[0],
                      "grade_1": "", "grade_2": "", "preferred": "", "comment": ""})
        key.append({"row": k + 1, "item_id": x["id"], "answer_1": first[1], "answer_2": second[1]})
    blind = f"results/blind_review_{COMPARE_WITH}_vs_{TAG}.csv"
    keyf = f"results/blind_key_{COMPARE_WITH}_vs_{TAG}.csv"
    for path, rows in ((blind, sheet), (keyf, key)):
        with open(path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    upload([(blind, blind), (keyf, keyf)], "04_eval: blind review sheet + key")
    print(f"{len(sheet)} rows -> {blind} (give to reviewer) | key -> {keyf} (keep private)")

## Done when
* **Mon 5 Oct:** `results/scores_base.csv` exists; commit it to GitHub **before** training starts
  (the commit time proves the baseline came first).
* **Wed 7 Oct:** `results/val_runA.csv`, `val_runB.csv`, `val_runC.csv`: choose the run with the best
  `action_ok` and lowest `under_referral` on validation.
* **Thu 8 Oct:** `scores_ilera.csv`, `compare_base_vs_ilera.csv` (the central results table for the
  submission and paper) and the blind-review sheet sent to the clinical reviewer.